In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from joblib import Parallel, delayed
from scipy.stats import entropy
from google.colab import files

warnings.filterwarnings("ignore")

# ==========================================
# 1. 実証的キャリブレーションと検証設定
# ==========================================
# 試行回数: Colab環境で現実的な時間でMCSEを収束させるための設定
# ※計算リソースが許せば10000〜10000000にスケールアップ可能
N_TRIALS = 1000

# 検証する次元数 D (属性の組み合わせ数)
DIMENSIONS = [10, 50, 100, 200]
# 標本サイズ N の対数グリッド (10^3 から 10^7)
SAMPLE_SIZES = np.logspace(3, 7, num=10, dtype=int)

# 同一統計集団とみなすKLダイバージェンスの許容閾値
EPSILON_THRESHOLD = 0.001

output_dir = "hypothesis3_integrated_collapse"
os.makedirs(output_dir, exist_ok=True)

print(f"仮説3（複合要因による構造崩壊）の実証シミュレーションを開始します...")
print(f"各 (D, N) の組み合わせにつき {N_TRIALS} 回の並列試行を実行します。")

# ==========================================
# 2. 単一試行関数 (H1 + H2 の統合DGP)
# ==========================================
def simulate_h3_single_trial(D, N):
    K = 5 # 5件法

    # 【H1要素: スパース化】セルのベース確率を初期値非依存(U(0,0.2))で生成
    cell_raw = np.random.uniform(0, 0.2, D)
    P_cell = cell_raw / np.sum(cell_raw)

    # 【H2要素: 認知バイアス】各セルに対して独立した確信度βを割り当て
    # 認知的霧(低β)から同調圧力(高β)までを含む社会の異質性を再現
    betas = 10 ** np.random.uniform(-1, 1.5, D)

    # 選択肢のベース嗜好も初期値非依存で生成
    X_choice = np.random.uniform(0, 0.2, (D, K))

    # 結合確率テンソルの生成
    P_joint = np.zeros((D, K))
    for d in range(D):
        # Softmax関数による確率変換
        exp_val = np.exp(betas[d] * X_choice[d])
        P_choice_given_d = exp_val / np.sum(exp_val)
        P_joint[d, :] = P_cell[d] * P_choice_given_d

    P_flat = P_joint.flatten()

    # 経験的分布のサンプリング (Multinomial)
    C_flat = np.random.multinomial(N, P_flat)
    Q_flat = C_flat / N

    # KLダイバージェンス計算のためのゼロ割り回避 (スムージング)
    Q_flat = np.where(Q_flat == 0, 1e-10, Q_flat)

    # 情報理論的距離の計算
    kl_div = entropy(P_flat, Q_flat)

    # 閾値を超えた場合、真の集団を捕捉できなかった(Collapse)と判定
    is_collapsed = 1 if kl_div > EPSILON_THRESHOLD else 0

    return [kl_div, is_collapsed]

# ==========================================
# 3. メイン・グリッド探索ループ
# ==========================================
results_summary = []

for D in DIMENSIONS:
    print(f"\n--- 次元数 D = {D} の検証 ---")
    for N_samp in SAMPLE_SIZES:
        # 並列処理によるモンテカルロ・シミュレーション
        trial_results = Parallel(n_jobs=-1)(
            delayed(simulate_h3_single_trial)(D, N_samp) for _ in range(N_TRIALS)
        )
        trial_results = np.array(trial_results)

        kl_95th = np.percentile(trial_results[:, 0], 95)
        collapse_count = np.sum(trial_results[:, 1])
        collapse_rate = collapse_count / N_TRIALS

        # MCSE (モンテカルロ標準誤差) による精度保証
        mcse = np.sqrt((collapse_rate * (1 - collapse_rate)) / N_TRIALS)

        results_summary.append({
            "Dimension_D": D,
            "Sample_Size_N": N_samp,
            "KL_Divergence_95th": kl_95th,
            "Collapse_Rate": collapse_rate,
            "MCSE": mcse
        })
        print(f" N={N_samp:<9} | KL(95%)={kl_95th:.5f} | Collapse Rate={collapse_rate*100:>5.1f}% | MCSE={mcse:.4f}")

df_res = pd.DataFrame(results_summary)
df_res.to_csv(f"{output_dir}/h3_integrated_collapse_metrics.csv", index=False)

# ==========================================
# 4. 限界標本サイズ N* の算出
# ==========================================
n_star_data = []
for D in DIMENSIONS:
    subset = df_res[df_res["Dimension_D"] == D]
    valid_n = subset[subset["Collapse_Rate"] < 0.05]["Sample_Size_N"] # 崩壊率5%未満のN

    n_star = valid_n.min() if not valid_n.empty else np.nan
    n_star_data.append({"Dimension_D": D, "Critical_Sample_Size_N_star": n_star})

df_n_star = pd.DataFrame(n_star_data)
df_n_star.to_csv(f"{output_dir}/h3_critical_N_star.csv", index=False)

# ==========================================
# 5. 可視化 (グラフの描画)
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid')

# 【グラフ1】サンプルサイズに対する崩壊率の推移とMCSE保証
plt.figure(figsize=(10, 6))
for D in DIMENSIONS:
    subset = df_res[df_res["Dimension_D"] == D]
    plt.errorbar(subset["Sample_Size_N"], subset["Collapse_Rate"],
                 yerr=1.96*subset["MCSE"], marker='o', capsize=4, label=f'Dimension D={D}')

plt.axhline(y=0.05, color='r', linestyle='--', label='Robust Threshold (5% Failure)')
plt.xscale('log')
plt.title("Probability of Statistical Collapse by Sample Size (with 95% CI)")
plt.xlabel("Sample Size N (log scale)")
plt.ylabel("Collapse Rate (Probability of KL > Threshold)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_collapse_rate_by_N.png", dpi=300)
plt.close()

# 【グラフ2】臨界サンプルサイズ N* の非線形爆発 (H3の実証)
plt.figure(figsize=(8, 6))
valid_n_star = df_n_star.dropna()
plt.plot(valid_n_star["Dimension_D"], valid_n_star["Critical_Sample_Size_N_star"],
         marker='s', color='purple', linewidth=2, markersize=8, label='H3: Integrated $N^*$ (Simulation)')

if not valid_n_star.empty:
    base_D = valid_n_star["Dimension_D"].iloc[0]
    base_N = valid_n_star["Critical_Sample_Size_N_star"].iloc[0]
    theoretical_N = base_N * (np.array(DIMENSIONS) / base_D)
    plt.plot(DIMENSIONS, theoretical_N, linestyle='--', color='gray', label=r'H1 Baseline: Theoretical $\mathcal{O}(D)$ scale')

plt.xscale('log')
plt.yscale('log')
plt.title("Non-linear Explosion of Critical Sample Size $N^*$ under H3")
plt.xlabel("Categorical Dimension D (log scale)")
plt.ylabel("Critical Sample Size $N^*$ (log scale)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig2_N_star_explosion.png", dpi=300)
plt.close()

# ==========================================
# 6. ZIP化とダウンロード
# ==========================================
zip_filename = "hypothesis3_integrated_collapse.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            file_path = os.path.join(root, file)
            zipf.write(file_path, arcname=file)

print(f"\n全検証が完了しました。結果を圧縮しました: {zip_filename}")
files.download(zip_filename)

仮説3（複合要因による構造崩壊）の実証シミュレーションを開始します...
各 (D, N) の組み合わせにつき 1000 回の並列試行を実行します。

--- 次元数 D = 10 の検証 ---
 N=1000      | KL(95%)=0.10976 | Collapse Rate=100.0% | MCSE=0.0000
 N=2782      | KL(95%)=0.02578 | Collapse Rate=100.0% | MCSE=0.0000
 N=7742      | KL(95%)=0.00624 | Collapse Rate=100.0% | MCSE=0.0000
 N=21544     | KL(95%)=0.00160 | Collapse Rate= 71.8% | MCSE=0.0142
 N=59948     | KL(95%)=0.00056 | Collapse Rate=  0.2% | MCSE=0.0014
 N=166810    | KL(95%)=0.00020 | Collapse Rate=  0.0% | MCSE=0.0000
 N=464158    | KL(95%)=0.00007 | Collapse Rate=  0.0% | MCSE=0.0000
 N=1291549   | KL(95%)=0.00003 | Collapse Rate=  0.0% | MCSE=0.0000
 N=3593813   | KL(95%)=0.00001 | Collapse Rate=  0.0% | MCSE=0.0000
 N=10000000  | KL(95%)=0.00000 | Collapse Rate=  0.0% | MCSE=0.0000

--- 次元数 D = 50 の検証 ---
 N=1000      | KL(95%)=0.81068 | Collapse Rate=100.0% | MCSE=0.0000
 N=2782      | KL(95%)=0.16823 | Collapse Rate=100.0% | MCSE=0.0000
 N=7742      | KL(95%)=0.03808 | Collapse Rate=100.0% | MCSE=

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>